# DICE — Cahier 1.3 (Incertitude, Monte Carlo) — Version étudiante

## 0) Installation et importation

In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120
RNG = np.random.default_rng(42)

def clone_params(base, **overrides):
    candidate = Params()
    for name, value in vars(base).items():
        setattr(candidate, name, value)
    for name, value in overrides.items():
        setattr(candidate, name, float(value))
    return candidate

def simulate(base, **overrides):
    candidate = clone_params(base, **overrides)
    path = init_states(candidate)
    path[1:, candidate.i_s] = 0.20
    path[1:, candidate.i_mu] = np.linspace(0.03, 0.60, candidate.nT - 1)
    path = update_path(path, range(1, candidate.nT), candidate)
    return path, candidate

def damage_fraction(path, par):
    lagged_temperature = np.r_[path[0, par.i_T_AT], path[:-1, par.i_T_AT]]
    damages = par.a2 * lagged_temperature ** par.a3
    if par.a4 != 0:
        damages += np.where(lagged_temperature > par.a6,
                            par.a4 * lagged_temperature ** par.a5, 0.0)
    return damages

def quantile_bands(array):
    return np.quantile(np.asarray(array), [0.05, 0.50, 0.95], axis=0)


## Q1) Simuler la simulation de base (un scénario).

Considérons un scénario de base. Simuler le modèle DICE avec l'étalonnage de base en tenant compte des commandes (exogènes): épargne constante`s_t = 0.20`; abatement `mu_t` rampes 0,03 -> 0,60 en 2060. Voie de déclaration (`T_AT`) et (`Y`) .

In [ ]:
# simulate(...) fixes the saving and abatement paths described above.
p = Params()
baseline, _ = simulate(p)
years = baseline[:, p.i_time]

# Plot baseline[:, p.i_T_AT] and baseline[:, p.i_Y] in two panels.


## Q2) Considérez maintenant l'incertitude paramétrique concernant la sensibilité à la température T2XCO2.

In Nordhaus (2018), one can read that:  
> *Équilibre de sensibilité à la température (ETS). La distribution de l'ETS adopte l'approche utilisée dans l'étude MUP. Les estimations primaires proviennent d'Olsen et al. (2012). Cette étude utilise une approche bayésienne, avec un précédent basé sur des études antérieures et une probabilité basée sur des données d'observation ou de modélisation. La meilleure distribution de montage est un PDF log-normal. Les paramètres de la distribution log-normale correspondent à Olsen et al. sont μ = 1,107 et ε = 0,264. Les principales statistiques sommaires de la distribution de référence dans l'étude sont les suivantes : moyenne = 3,13, médiane = 3,03 et écart type = 0,843.


### Q2A) Tirer 1 000 observations de`T2XCO2` et signaler l'histogramme.

*Hints:*
- Utiliser une distribution lognormale centrée sur la base`p.T2XCO2`.
- Set `sigma_ecs = math.log(1.25) / norm.ppf(0.95)`.
- Dessiner avec`np.exp(math.log(p.T2XCO2) + sigma_ecs * RNG.standard_normal(N))`.
- Placez l'histogramme et signalez les quantiles de 5%, 50% et 95%.


In [ ]:
N = 1000
sigma_ecs = math.log(1.25) / norm.ppf(0.95)
# ecs_draws = ...


### Q2B) Créez une boucle qui génère le chemin de mise à jour et les stocke.
*Hints:*
- Commencez par`p = Params()`, `sim = init_states(p)`.  
- Pour chaque dessin, créez un nouvel objet de paramètre et remplacez :`p2.T2XCO2 = float(t2x)`.  
- Régler les contrôles comme au départ (constante`s`, rampe linéaire pour`mu`).  
- Run `update_path(sim2, range(1, p2.nT), p2)`.  
- Store results in lists, e.g. `T_list.append(sim2[:, p2.i_T_AT])`, `Y_list.append(sim2[:, p2.i_Y])`, et dommages`D_list`  (compute it manually).


In [ ]:
# Your code for question 2-B should be here


### Q2C) Indiquer l'intervalle de confiance pour les températures, le PIB et les dommages.
*Hints:*
- Utilisation`np.quantile(arr, [0.05,0.5,0.95], axis=0)` pour calculer les bandes de 5 à 50 à 95 %.
- Graphiques de ventilateur de parcelle avec`plt.fill_between(years, q05, q95, alpha=0.2)` et`plt.plot(years, q50)`.  
- Extraire les valeurs en 2100 avec`i2100 = np.argmin(np.abs(years-2100))` et imprimer`q05[i2100], q50[i2100], q95[i2100]`.  


In [ ]:
# Your code for question 2-C should be here

> ✍️ You written answer here.

## Q3) Considérez maintenant l'incertitude paramétrique du paramètre de dommage a2.

In Nordhaus (2018), one can read that:  
> *=J'ai réglé, avec les différentes approches, une valeur pour l'incertitude du paramètre de dommage qui est la moitié de la valeur moyenne du paramètre. Plus précisément, la distribution est supposée normale, avec un écart-type de 0,118 % Y/°C2. Ceci reflète la grande divergence actuelle entre les différentes études.



### Q3A) Dessiner 1 000 observations du paramètre de dommage`a2` et signaler l'histogramme.

*Hints:*
- Utiliser une distribution lognormale positive centrée sur`p.a2`.
- Set `sigma_a2 = math.log(2.0) / norm.ppf(0.95)`.
- Dessiner avec`np.exp(math.log(p.a2) + sigma_a2 * RNG.standard_normal(N))`.
- Placez l'histogramme et signalez les quantiles de 5%, 50% et 95%.


In [ ]:
# Your code for question 3 should be here

### Q3B) Simuler l'incertitude articulaire dans la sensibilité au climat et les dommages.

*Hints:*
- Loop over `zip(ecs_draws, a2_draws)`.
- Pour chaque paire, appelez`simulate(p, T2XCO2=ecs, a2=a2)`.
- Température, sortie et`damage_fraction(path, par)`.
- Convertir les trois listes en tableaux NumPy avant de calculer les quantiles.


In [ ]:
# Your code for question 3 should be here

### Q3C) Indiquer l'intervalle de confiance pour les températures, le PIB et les dommages. Interprétation.
*Hints:*
- Utilisation`np.quantile(arr, [0.05,0.5,0.95], axis=0)` l'ensemble simulé.
- Graphiques de ventilateur de parcelle avec`plt.fill_between(years, q05, q95, alpha=0.2)` et`plt.plot(years, q50)`.  
- Valeurs de rapport pour 2100: trouver l'index avec`i2100 = np.argmin(np.abs(years-2100))` et imprimer les quantiles pour`T_AT`, `Y`Et des dommages.

In [ ]:
# Your code for question 3 should be here

> ✍️ You written answer here.

### Q3D) Comparer l'incertitude du SCE seulement avec l'incertitude du SCE mixte + dommages.

Placez les deux bandes de température de 5 à 95 % sur les mêmes axes. Alors comparez leur
widths in 2100 using `q95 - q05` pour la température, la sortie et les dommages.


In [ ]:
# Your code for question 3 should be here

> ✍️ You written answer here.

## Q4) Considérez maintenant l'incertitude paramétrique au sujet du paramètre de décarbonisation (t).

In Nordhaus (2018), one can read that:  
> *La méthode la plus simple consiste à estimer une régression OLS en utilisant les données de 1960 à 2015, puis à examiner l'erreur de prévision pour 2100. Si un terme AR1 est inclus dans l'équation, l'erreur type de la prévision pour 2100 est de 13,5 % du logarithme de ε(t). Cela implique une incertitude annuelle de 0,149 % par an. Cependant, une racine unitaire de φ(t) ne peut pas être rejetée, de sorte que cette estimation est biaisée vers le bas.


### Q4A) Dessinez 1 000 observations du paramètre de tendance à la décarbonisation.

Par défaut`deltasig` est zéro. Pour cet exercice de stress transparent, utilisez un
positive half-normal distribution:

```python
deltasig_draws = np.abs(RNG.normal(loc=0.0, scale=0.02, size=N))
```

Placez l'histogramme et signalez les quantiles de 5%, 50% et 95%.


In [ ]:
# Your code for question 4 should be here

### Q4B) Simuler conjointement les trois sources d'incertitude.

*Hints:*
- Loop over `zip(ecs_draws, a2_draws, deltasig_draws)`.
- Call `simulate(p, T2XCO2=ecs, a2=a2, deltasig=deltasig)`.
- Entreposer la température, la sortie et les dommages exactement comme dans Q3B.


In [ ]:
# Your code for question 4 should be here

### Q4C) Indiquer l'intervalle de confiance pour les températures, le PIB et les dommages.
*Hints:*
- Utilisation`np.quantile(arr, [0.05,0.5,0.95], axis=0)` pour calculer les bandes de confiance dans l'ensemble.
- Graphiques de ventilateur de parcelle avec`plt.fill_between(years, q05, q95, alpha=0.2)` et couvrir la médiane avec`plt.plot(years, q50)`.  
- Extraire 2100 valeurs avec`i2100 = np.argmin(np.abs(years-2100))` et imprimez la plage de 5 à 50 à 95 %.

In [ ]:
# Your code for question 4 should be here

In [ ]:
# Intentionally left as a workspace cell.